Setup the basics


In [41]:
from google.cloud import bigquery
from google import genai
from google.genai.types import GenerateContentConfig

PROJECT_ID = "qwiklabs-gcp-04-238cff0c99bd"
REGION     = "us-central1"
DATASET    = "aurora_bay"

bq = bigquery.Client(project=PROJECT_ID)
enai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

Create the dataset, then load directly from the public GCS path using autodetect schema.




In [42]:
from google.cloud import bigquery

ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = REGION          # "us-central1" — must match the model/connection
bq.create_dataset(ds, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET} in {REGION}")

job = client.load_table_from_uri(
    "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv",
    f"{PROJECT_ID}.{DATASET}.faqs",
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1, autodetect=True,
        write_disposition="WRITE_TRUNCATE",
    ),
)
job.result()

Dataset ready: qwiklabs-gcp-04-238cff0c99bd.aurora_bay in us-central1


LoadJob<project=qwiklabs-gcp-04-238cff0c99bd, location=us-central1, id=fdc5741d-04ea-42e0-9da8-11fb76b6eade>

Setup BQ data connection - this will error if already setup - harmless so ignore

In [43]:
!bq mk --connection --location={REGION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE embeddings_conn || echo "connection already exists"n

BigQuery error in mk operation: Already Exists: Connection
projects/875877181246/locations/us-central1/connections/embeddings_conn
connection already existsn


Setup IAM service account

In [44]:
SA = "bqcx-875877181246-phs9@gcp-sa-bigquery-condel.iam.gserviceaccount.com"

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{SA}" \
    --role="roles/aiplatform.user"

Updated IAM policy for project [qwiklabs-gcp-04-238cff0c99bd].
bindings:
- members:
  - serviceAccount:service-875877181246@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-875877181246@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-875877181246@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-875877181246-phs9@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-04-238cff0c99bd@qwiklabs-gcp-04-238cff0c99bd.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:875877181246@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-875877181246@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- m

Create the embeddings

In [45]:
bq.query(f"""
CREATE OR REPLACE MODEL `{DATASET}.embedding_model`
REMOTE WITH CONNECTION `{REGION}.embeddings_conn`
OPTIONS (ENDPOINT = 'text-embedding-005')
""").result()

In [46]:
bq.query(f"""
CREATE OR REPLACE TABLE `{DATASET}.faqs_embedded` AS
SELECT
  question,
  answer,
  ml_generate_embedding_result
FROM ML.GENERATE_EMBEDDING(
  MODEL `{DATASET}.embedding_model`,
  (SELECT
     string_field_0 AS question,
     string_field_1 AS answer,
     CONCAT(string_field_0, ' ', string_field_1) AS content
   FROM `{DATASET}.faqs`),
  STRUCT(TRUE AS flatten_json_output)
)
""").result()
print("Embeddings table created: faqs_embedded")

Embeddings table created: faqs_embedded


Setup a test method rather than spin up a chat bot

In [47]:
from google.cloud import bigquery

def retrieve(question, top_k=3):
    """Embed the question and return the top_k most similar FAQ rows."""
    sql = f"""
    SELECT base.question, base.answer, distance
    FROM VECTOR_SEARCH(
      TABLE `{DATASET}.faqs_embedded`, 'ml_generate_embedding_result',
      (SELECT ml_generate_embedding_result FROM ML.GENERATE_EMBEDDING(
          MODEL `{DATASET}.embedding_model`,
          (SELECT @q AS content),
          STRUCT(TRUE AS flatten_json_output))),
      top_k => {top_k}, distance_type => 'COSINE')
    ORDER BY distance
    """
    params = [bigquery.ScalarQueryParameter("q", "STRING", question)]
    return bq.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

Method to ask a question - takes the original question and searches the embeddings for matches - takes the top 3 and rewrites the promp with an FAQ section to ground the LLM, tells the llm to use the FAQ answers to answer the user entered questions then e,beds the original user question before submitting to gemini.

In [48]:
from google import genai
from google.genai.types import GenerateContentConfig

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

def ask(question, top_k=3):
    """Retrieve context, ground Gemini on it, return (answer, context_df)."""
    ctx = retrieve(question, top_k)
    context = "\n\n".join(f"Q: {r.question}\nA: {r.answer}" for _, r in ctx.iterrows())
    prompt = f"""You are the Aurora Bay FAQ assistant. Answer using ONLY the FAQs provided.
If the answer isn't found, say you don't have that information.

FAQs:
{context}

User question: {question}"""
    resp = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=GenerateContentConfig(temperature=0.2))
    return resp.text, ctx

Ask some questions - 1 and 2 are the same but worded differently and the last one has no bearing on the data set. Noticed that the distance was returned so outputted it for my own curiosity. Question 2 was cool because we didnt have an exact match but a close one that was the same thing.

In [50]:
sample_queries = [
    "When was Aurora Bay founded?",
    "What year was Aurora Bay founded?",
    "What is the population of Aurora Bay?",
    "Where is the Town Hall located?",
    "What is there to do in Aurora Bay?",
    "How do I contact the town offices?",
    "How far away is the moon?"
]

for i, q in enumerate(sample_queries, start=1):
    answer, ctx = ask(q)
    print("=" * 70)
    print(f"QUESTION: {q}")
    print("-" * 70)
    print(f"ANSWER:\n{answer}")
    print("-" * 70)
    top = ctx.iloc[0]
    print(f"TOP MATCH (distance={top.distance:.4f}): {top.question}")
    print("=" * 70 + "\n")

QUESTION: When was Aurora Bay founded?
----------------------------------------------------------------------
ANSWER:
Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.
----------------------------------------------------------------------
TOP MATCH (distance=0.1669): When was Aurora Bay founded?

QUESTION: What year was Aurora Bay founded?
----------------------------------------------------------------------
ANSWER:
Aurora Bay was founded in 1901.
----------------------------------------------------------------------
TOP MATCH (distance=0.1765): When was Aurora Bay founded?

QUESTION: What is the population of Aurora Bay?
----------------------------------------------------------------------
ANSWER:
Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.
----------------------------------------------------------------------
TOP MATC